# check the catalog and schema

In [0]:
%sql
Select current_metastore(), current_catalog(), current_schema()

In [0]:
#python equivalent
result = spark.sql("Select current_metastore(), current_catalog(), current_schema()")
display(result)

# Step1: Create Sample Data

In [0]:
from pyspark.sql import SparkSession
sprak=SparkSession.builder.appName("churn_mlops").getOrCreate()

data=[
    (1, 24, 20000, 0),
    (2, 45, 50000, 1),
    (3, 30, 30000, 0),
    (4, 50, 70000, 1),
    (5, 35, 40000, 0),
]

columns=["customer_id", "age", "income", "churn"]

df=spark.createDataFrame(data, columns)

df.write.format("delta").mode("overwrite").saveAsTable("churn_raw")

# Step2: Feature Enginnering Notebook

In [0]:
df_raw=spark.read.table("churn_raw")

from pyspark.ml.feature import VectorAssembler

feature_cols=["age", "income"]

assembler=VectorAssembler(inputCols=feature_cols, outputCol="features")

df_features=assembler.transform(df_raw)

df_features.write.format("delta").mode("overwrite").saveAsTable("churn_features")

                         

# Step3: Model training with MLFlow

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.classification import LogisticRegression
from mlflow.models import infer_signature

df_churn_feature=spark.read.table("churn_features")

mlflow.end_run()
with mlflow.start_run() as run:
    lr=LogisticRegression(
        featuresCol="features",
        labelCol="churn"
    )

    model=lr.fit(df_churn_feature)
    predictions = model.transform(df_churn_feature)
    signature = infer_signature(df_churn_feature.toPandas(), predictions.select("prediction").toPandas())

    curr_catalog="workspace"
    curr_schema="default"
    volume_name="aiml"

    mlflow.spark.log_model(model, "chrun_model", dfs_tmpdir=f"/Volumes/{curr_catalog}/{curr_schema}/{volume_name}/mlflow_tmp", signature=signature)

    mlflow.log_param("model", "Logistic Regression")
    summary=model.summary
    mlflow.log_metric("accuracy", summary.accuracy)
    run_id = run.info.run_id

# Step4: Register Model

In [0]:
import mlflow

model_uri=f"runs:/{run_id}/chrun_model"

mlflow.register_model(model_uri, "churn_prediction_model")

# trackering in github

In [0]:
#this is part of main
#this has been added in feature1